专利选股策略——回测优化与行业中性化

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import os
import streamlit as st
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tabulate import tabulate

# 一、前置处理：专利因子构建与面板数据构建
# 1.1 导入库与数据路径配置
data_path = 'D:/Limmo/Python数分/实验数据/'
db = duckdb.connect()
cache_path = 'D:/Limmo/Python数分/实验四/'

# 读取三大核心数据集
if os.path.exists(f'{cache_path}daily_cleaned.parquet'):
    daily = pd.read_parquet(f'{cache_path}daily_cleaned.parquet')
else:
    daily = db.execute(''' SELECT SPLIT_PART("股票代码", '.', 1) AS 股票代码, "交易日", "收盘价", "涨跌幅（%）" AS 涨跌幅,
                        "总市值（万元）" AS 总市值, "流通市值（万元）" AS 流通市值 FROM read_parquet(?) ''', 
                        [f'{data_path}股票日线全量数据.parquet']).df()
    daily.to_parquet(f'{cache_path}daily_cleaned.parquet')
print(f"daily: {len(daily)} 行, {len(daily.columns)} 列")

if os.path.exists(f'{cache_path}patent_cleaned.parquet'):
    patent = pd.read_parquet(f'{cache_path}patent_cleaned.parquet')
else:
    patent = db.execute(''' SELECT "关联股票代码" AS 股票代码, "申请年份", "专利类型", "专利状态", "专利申请号"
                       FROM read_parquet(?) ''', [f'{data_path}上市公司专利.parquet']).df()
    patent['股票代码'] = patent['股票代码'].str.split('[').str[1].str.split('，').str[0].str.split(']').str[0]
    patent.to_parquet(f'{cache_path}patent_cleaned.parquet')
print(f"patent: {len(patent)} 行, {len(patent.columns)} 列")

finance = db.execute(''' SELECT "Stkcd" AS 股票代码, "Accper" AS 年份, "Typrep", "A001000000", "F050201B", "Indcd", "Indnme"  FROM read_parquet(?) ''', [f'{data_path}上市公司财报.parquet']).df()
print(f"finance: {len(finance)} 行, {len(finance.columns)} 列")

patent0 = len(patent)
daily0 = len(daily)
# 专利去重：按股票代码+申请年份+专利申请号
patent = patent.drop_duplicates(subset=['股票代码', '申请年份', '专利申请号'])
print(f'专利去重: {patent0} → {len(patent)}（去除{patent0-len(patent)}条重复，{((patent0-len(patent))/patent0):.1%}）')
patent1 = len(patent)
patent = patent.dropna(subset=['股票代码', '申请年份', '专利申请号'])
print(f'专利缺失值过滤: {patent1} → {len(patent)}（去除{patent1-len(patent)}条）')
# 缺失值过滤
daily = daily.dropna(subset=['收盘价', '涨跌幅', '总市值'])
print(f'日线缺失值过滤: {daily0} → {len(daily)}（去除{daily0-len(daily)}条）')

# 1.2 候选池与基准数据加载
zz2000 = db.execute(''' SELECT * FROM read_csv(?) ''', [f'{data_path}中证2000成分股.csv']).df()
zz2000_codes = set(zz2000['成分券代码'].astype(str).str.zfill(6))
print(f'\n中证2000成分股：{len(zz2000)}只')
print(f'中证2000代码集：{len(zz2000_codes)}个')

st_stocks = db.execute(''' SELECT * FROM read_csv(?) ''', [f'{data_path}ST股列表.csv']).df()
st_codes = set(st_stocks['股票代码'].astype(str).str.split('.').str[0])
print(f'ST股：{len(st_stocks)}只')
print(f'ST代码集：{len(st_codes)}个')

# 读取基准指数
benchmark = pd.read_excel(f'{data_path}932000.xlsx', header=2)
benchmark = benchmark.iloc[:3016]
benchmark = benchmark.drop(index=1)
benchmark.columns = benchmark.columns.str.strip()
benchmark['交易日'] = pd.to_datetime(benchmark['时间'])
benchmark['指数日收益率'] = benchmark['收盘'].pct_change()
benchmark['指数日收益率'] = benchmark['指数日收益率'].fillna(0)
benchmark = benchmark[['交易日', '收盘', '指数日收益率']]
benchmark.set_index('交易日', inplace=True)
print(f'中证2000日线：{len(benchmark)}条')

# 1.3 财报因子与公司成立日期处理
establish = pd.read_excel(f'{data_path}公司成立日期.xlsx')
establish = establish.rename(columns={'证券代码': '股票代码'})
# 筛选年报数据
finance = finance[(finance['Typrep'] == 'A') & (finance['年份'].str.endswith('12-31'))]
finance['年份'] = pd.to_datetime(finance['年份'])
finance = finance[(finance['年份'].dt.year >= 2010) & (finance['年份'].dt.year <= 2024)]
print(f'\n财报因子：{len(finance)}条\n公司成立日期: {len(establish)}条')
# 类型转换
finance['年份'] = finance['年份'].dt.year

# 1.4 六维度数据质量检查报告
print("\n" + "="*50)
print("1.4 数据质量检查报告")
print("="*50)

# 1.4.1. 数据规模
print(f"\n1. 数据规模")
print(f"daily: {len(daily)} 行, {len(daily.columns)} 列")
print(f"patent: {len(patent)} 行, {len(patent.columns)} 列")

# 1.4.2. 时间范围
print(f"\n2. 关键字段时间范围")
print(f"daily.交易日: {daily['交易日'].min().date()} 至 {daily['交易日'].max().date()}")
print(f"patent.申请年份: {patent['申请年份'].min()} 至 {patent['申请年份'].max()}")

# 1.4.3. 缺失率 (清洗后应全部为 0%)
print(f"\n3. 关键字段缺失率")
for col in ['收盘价', '总市值', '涨跌幅']:
    missing_rate = daily[col].isnull().mean() * 100
    print(f"daily.{col}:缺失率 {missing_rate:.2f}%")
print(f"patent.关联股票代码：缺失率 {patent['股票代码'].isnull().mean():.2%}")

# 1.4.4. 重复记录检查
print(f"\n4. 重复记录检查")
daily_dup = daily.duplicated(subset=['股票代码', '交易日']).sum()
print(f"daily: 总行数={len(daily)}, 唯一行={len(daily)-daily_dup}, 重复行: {daily_dup} ({(daily_dup/len(daily)):.2%})")
patent_dup = patent.duplicated(subset=['股票代码', '申请年份', '专利申请号']).sum()
print(f"patent: 总行数={len(patent)}, 唯一行={len(patent)-patent_dup}, 重复行: {patent_dup} ({(patent_dup/len(patent)):.2%})")

# 1.4.5. 异常值检查 (收盘价<=0, 涨跌幅>20%)
print(f"\n5. 关键数值字段异常值")
bad_close = daily[daily['收盘价'] <= 0].shape[0]
bad_ret = daily[daily['涨跌幅'].abs() > 20].shape[0]  # 超过20%的涨跌幅
print(f"daily ['收盘价'<=0]: {bad_close}/{len(daily)}({bad_close/len(daily):.2%})")
print(f"daily ['涨跌幅'>20%]: {bad_ret}/{len(daily)}({bad_ret/len(daily):.2%})")

# 1.4.6. 候选池交叉覆盖
print(f"\n6. 候选池交叉覆盖 (中证2000与日线、专利交集)")

in_daily = zz2000_codes.intersection(set(daily['股票代码'].unique()))
in_patent = zz2000_codes.intersection(set(patent['股票代码'].unique()))
print(f"中证2000成分股总数量: {len(zz2000_codes)}")
print(f"同时出现在日线数据中的数量: {len(in_daily)} ({len(in_daily)/len(zz2000_codes):.2%})")
print(f"同时出现在专利数据中的数量: {len(in_patent)} ({len(in_patent)/len(zz2000_codes):.2%})")
in_both = in_daily.intersection(in_patent)
print(f"同时出现在日线数据和专利数据中的数量: {len(in_both)} ({len(in_both)/len(zz2000_codes):.2%})")
print("\n" + "="*50)

# 1.5 专利因子与面板数据构建
print('\n1.5 专利因子与面板数据构建')
# 1.5.1 专利因子构建
patent['股票代码'] = patent['股票代码'].astype(str)
patent['申请年份'] = patent['申请年份'].astype(int)
patent = patent.rename(columns={'申请年份': '年份'})
patent = patent[(patent['年份'] >= 2014) & (patent['年份'] <= 2024)]
patent_agg = patent.groupby(['股票代码', '年份']).agg(
    patent_count=('专利申请号', 'count'),
    invention_count=('专利类型', lambda x: (x == '发明').sum()),
    authorized_count=('专利状态', lambda x: (x == '授权').sum())
).reset_index()
patent_agg['authorization_rate'] = patent_agg['authorized_count'] / patent_agg['patent_count']
patent_agg['invention_ratio'] = patent_agg['invention_count'] / patent_agg['patent_count']
print(f'专利聚合：{len(patent_agg)} 条 (股票×年份)')
print(f'股票数：{len(patent_agg.drop_duplicates(subset=['股票代码']))}')

# 1.5.2 日线数据聚合
daily['股票代码'] = daily['股票代码'].astype(str).str.zfill(6)
daily['交易日'] = pd.to_datetime(daily['交易日'])
daily = daily.set_index('交易日').sort_index()
daily['年份'] = daily.index.year
daily = daily[(daily['年份'] >= 2010) & (daily['年份'] <= 2024)]
vol_mv = daily.groupby(['股票代码', '年份']).agg({'收盘价': 'mean', '涨跌幅': 'std', 
                                            '总市值': 'mean', '流通市值':'mean'}).reset_index()

vol_mv.columns = ['股票代码', '年份', 'avg_close', 'volatility', 'avg_total_mv', 'avg_circ_mv']
#计算交易天数
trade_days = daily.groupby(['股票代码', '年份']).size().reset_index(name='trade_days')
vol_mv = vol_mv.merge(trade_days, on=['股票代码', '年份'])
vol_mv = vol_mv[vol_mv['trade_days'] >= 100]
print(f'波动率+市值+收益率: {len(vol_mv)} 条')

# 1.6 合并面板与候选池标记
print('\n1.6 合并面板与候选池标记')
panel = pd.merge(patent_agg, vol_mv, on=['股票代码', '年份'], how='left')
print(f'合并面板: {len(panel)} 条')
panel['patent_per_mv'] = panel['patent_count'] / panel['avg_total_mv']
panel['in_pool'] = (panel['股票代码'].isin(zz2000_codes)) & (~panel['股票代码'].isin(st_codes))
print(f'中证2000+非ST: {panel['in_pool'].sum()} 条')
print(f'股票数: {len(panel.drop_duplicates(subset=['股票代码']))}')
print(f'年份: {panel['年份'].min()} ~ {patent['年份'].max()}')

# 二、Pandas低频回测：专利选股组合构建与绩效评估
# 2.1 按patent_per_mv选股 (Top100)
print('\n2.1 按patent_per_mv选股 (Top100)')
selected = panel[panel['in_pool']].copy()
selected = selected.sort_values(['年份', 'patent_per_mv'], ascending=[True, False])
selected = selected.groupby('年份').head(100).reset_index(drop=True)
print(f'选取结果: {len(selected)} 条, 每年100只')
print(selected['年份'].value_counts().sort_index())

# 2.2 构建组合日线数据
print('\n2.2 构建组合日线数据')
daily = daily.reset_index()
daily_selected = daily.merge(selected[['股票代码', '年份']], on=['股票代码', '年份'], how='inner')
daily_selected = daily_selected[daily_selected['年份'] == daily_selected['交易日'].dt.year]
print(f'\n组合日线数据: {len(daily_selected)} 条')

# 2.3 回测计算（净值、超额收益、回撤）
print('\n2.3 回测计算（净值、超额收益、回撤）')
port_daily = daily_selected.groupby('交易日').apply(
    lambda x: np.average(x['涨跌幅']/100, weights=x['总市值'])
).reset_index(name='port_ret')
port_daily = port_daily.set_index('交易日')
backtest = port_daily.join(benchmark['指数日收益率'], how='inner')

backtest = backtest.rename(columns={'指数日收益率': 'zz2000_ret'})
backtest = backtest.sort_index()
backtest['port_nav'] = (1 + backtest['port_ret']).cumprod()
backtest['zz2000_nav'] = (1 + backtest['zz2000_ret']).cumprod()
backtest['excess_ret'] = backtest['port_ret'] - backtest['zz2000_ret']
backtest['excess_nav'] = (1 + backtest['excess_ret']).cumprod()
backtest['port_peak'] = backtest['port_nav'].cummax()
backtest['port_drawdown'] = backtest['port_nav'] / backtest['port_peak'] - 1
backtest['zz2000_peak'] = backtest['zz2000_nav'].cummax()
backtest['zz2000_drawdown'] = backtest['zz2000_nav'] / backtest['zz2000_peak'] - 1
print(f"回测数据：{len(backtest)} 个交易日")
print(f"日期范围：{backtest.index.min().date()} ~ {backtest.index.max().date()}")
print("\n回测数据前5行预览：")
print(backtest[['port_ret', 'port_nav', 'zz2000_ret', 'zz2000_nav', 'excess_ret']].head())

# 2.4 计算回测绩效指标
print('\n2.4 计算回测绩效指标')
def calc_metrics(returns, benchmark_returns=None, freq=252):
    ann_ret = (1 + returns.mean()) ** freq - 1
    ann_vol = returns.std() * np.sqrt(freq)
    sharpe = (returns.mean() / returns.std()) * np.sqrt(freq)
    nav = (1 + returns).cumprod()
    peak = nav.cummax()
    dd = nav / peak - 1
    max_dd = dd.min()
    win_rate = (returns > 0).mean()
    metrics = {
        '年化收益': f"{ann_ret:.2%}",
        '年化波动': f"{ann_vol:.2%}",
        '夏普比率': f"{sharpe:.4f}",
        '最大回撤': f"{max_dd:.2%}",
        '胜率': f"{win_rate:.2%}"
    }
    if benchmark_returns is not None:
        excess = returns - benchmark_returns
        ir = (excess.mean() / excess.std()) * np.sqrt(freq)
        metrics['信息比率'] = f"{ir:.4f}"
        metrics['超额年化'] = f"{(1 + excess.mean()) ** freq - 1:.2%}"
    return metrics
print('\n' + '='*50)
print("专利选股组合(Top 100, 市值加权)")
print('='*50)
metrics_original = calc_metrics(backtest['port_ret'], backtest['zz2000_ret'])
for k, v in metrics_original.items():
    print(f"{k}: {v}")
print('\n' + '='*50)
print("中证2000基准")
print('='*50)
metrics_benchmark = calc_metrics(backtest['zz2000_ret'])
for k, v in metrics_benchmark.items():
    print(f"{k}: {v}")

# 2.5 Plotly可视化 三合一图表
print('\n2.5 Plotly可视化 三合一图表')
# 2.5.1. 创建画布：3行1列，共享X轴，设置子图高度比例
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     row_heights=[0.5, 0.25, 0.25],subplot_titles=("净值对比", "超额收益曲线", "回撤曲线"))
# 2.5.2 第1个图：净值对比
# 原始策略净值 (红色实线)
fig.add_trace(go.Scatter(x=backtest.index, y=backtest['port_nav'], mode='lines', name='专利选股组合', 
                line=dict(color='#d62728', width=1.5)), row=1, col=1)
# 中证2000净值 (蓝色实线)
fig.add_trace(go.Scatter(x=backtest.index, y=backtest['zz2000_nav'], mode='lines', name='中证2000',
                line=dict(color='#1f77b4', width=1.5)), row=1, col=1)

# 2.5.3第2个图：超额收益曲线
# 超额净值 (绿色实线)
fig.add_trace(go.Scatter(x=backtest.index, y=backtest['excess_nav'], mode='lines', name='超额净值',
                line=dict(color='#2ca02c', width=1.5)), row=2, col=1)
# 在y=1.0处画一条虚线，代表"刚好持平"
fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)
# 2.5.4第3个图：回撤曲线
# 策略回撤 (红色面积)
fig.add_trace(go.Scatter(x=backtest.index, y=backtest['port_drawdown'], mode='lines', fill='tozeroy', 
        name='组合回撤', line=dict(color='rgba(214, 39, 40, 0.8)', width=1), fillcolor='rgba(214, 39, 40, 0.8)'  # 半透明红色
    ), row=3, col=1)
# 基准回撤 (蓝色面积)
fig.add_trace(go.Scatter(x=backtest.index, y=backtest['zz2000_drawdown'], mode='lines', fill='tozeroy', 
        name='中证2000回撤', line=dict(color='rgba(31, 119, 180, 0.5)', width=1), fillcolor='rgba(31, 119, 180, 0.5)'  # 半透明蓝色
    ), row=3, col=1)
# 2.5.5 图表美化设置
for trace in fig.data:
    trace.visible = True
fig.update_yaxes(row=1, col=1, range=[0, 8])
fig.update_yaxes(row=2, col=1, tickformat='.2f')
fig.update_yaxes(row=3, col=1, tickformat='.2f')
fig.update_layout(height=800, width=1000, legend=dict(orientation="h", yanchor="bottom", y=1.02,
                    xanchor="right", x=1), margin=dict(l=50, r=50, t=80, b=50))
fig.show()

# 三、行业中性化选股：风暴因子暴露控制与对冲效果分析
print("\n3.1 行业分类映射 (CAPCO 二级分类)")
# 3.1 行业分类映射
SUB_CATEGORY_MAP = {
    13: 'CA', 14: 'CA', 15: 'CA',
    17: 'CB', 18: 'CB', 19: 'CB',
    20: 'CC', 21: 'CC', 22: 'CD', 23: 'CD', 24: 'CD',
    25: 'CE', 26: 'CE', 27: 'CF', 28: 'CG', 29: 'CG',
    30: 'CH', 31: 'CI', 32: 'CI', 33: 'CI', 34: 'CI', 35: 'CI',
    36: 'CK', 37: 'CK', 38: 'CL', 39: 'CL', 40: 'CL',
    41: 'CM', 42: 'CM', 43: 'CM'
}

SUB_CATEGORY_NAMES = {
    'CA': '食品饮料烟草', 'CB': '纺织服装鞋帽', 'CC': '木材家具',
    'CD': '造纸印刷文教', 'CE': '石油化学', 'CF': '医药',
    'CG': '化纤橡胶塑料', 'CH': '非金属', 'CI': '金属',
    'CJ': '通用专用设备', 'CK': '汽车船舶航空航天', 'CL': '电气计算机仪器仪表',
    'CM': '其他制造',
    'A': '农、林、牧、渔业', 'B': '采矿业', 'D': '电力、热力、燃气及水生产和供应业',
    'E': '建筑业', 'F': '批发和零售业', 'G': '交通运输、仓储和邮政业',
    'H': '住宿和餐饮业', 'I': '信息传输、软件和信息技术服务业', 'J': '金融业',
    'K': '房地产业', 'L': '租赁和商务服务业', 'M': '科学研究和技术服务业',
    'N': '水利、环境和公共设施管理业', 'O': '居民服务、修理和其他服务业',
    'P': '教育', 'Q': '卫生和社会工作', 'R': '文化、体育和娱乐业',
    'S': '综合', 'T': '国际组织'
}

finance['Indcd'] = finance['Indcd'].replace(['nan', 'None', 'NaN', ''], pd.NA)
clean_finance = finance.dropna(subset=['Indcd']).copy()
print(f"原始表行数: {len(finance)}, 有行业代码的行数: {len(clean_finance)}")
clean_finance = clean_finance.sort_values('年份', ascending=False)
industry_map = clean_finance.drop_duplicates(subset='股票代码')[['股票代码', 'Indcd']].copy()

def get_sub_category(code):
    code = str(code).strip()
    if not code: return 'X'
    if code.startswith('C') and len(code) >= 3:
        try:
            num = int(code[1:4])
            return SUB_CATEGORY_MAP.get(num, 'C')
        except:
            return 'C'
    return code[0]

industry_map['sub_category'] = industry_map['Indcd'].apply(get_sub_category).fillna('X')
print(f"行业映射: {len(industry_map)} 只股")
print("次类代码分布:")
print(industry_map['sub_category'].value_counts(), None)

# 3.2 合并小行业组
pool = panel[panel['in_pool']].copy()
pool = pool.merge(industry_map[['股票代码', 'sub_category']], on='股票代码', how='left')
pool['sub_category'] = pool['sub_category'].fillna('X').replace({'None': 'X', 'N': 'X'})
# 全局判断，确保跨年标签一致
global_counts = pool.groupby('sub_category')['股票代码'].nunique()
# 找出不够 100 只的次类（比如 CA, CB, CC 这些）
small_groups = global_counts[global_counts < 100].index.tolist()
# 执行合并：对不够 100 只的进行归类
def merge_small_groups(cat):
    if cat in small_groups:
        return cat[0]
    return cat
pool['industry_label'] = pool['sub_category'].apply(merge_small_groups)
# 打印合并后的结果
print(f"\n3.2 合并后分组标签 (示例年份2024):")
temp = pool[pool['年份']==2024]['industry_label'].value_counts()
clean_list = temp[temp.index != 'X'].sort_values(ascending=False)
for code, count in clean_list.items():
    ch_name = SUB_CATEGORY_NAMES.get(code, '')
    if ch_name:
        print(f"{code}({ch_name}): {count}只")
    else:
        print(f"{code}: {count}只")

# 3.3 计算各组市值占比
if '总市值' in pool.columns:
    pool = pool.rename(columns={'总市值': 'avg_total_mv'})
# 计算每年每个行业的“组总市值”
group_mv = pool.groupby(['年份', 'industry_label'])['avg_total_mv'].sum().reset_index()
# 计算每年全市场的“总市值”
total_mv_per_year = pool.groupby('年份')['avg_total_mv'].sum().reset_index()
# 将两者合并，计算目标权重（占比），左表是组总市值，右表是年总市值
group_weights = group_mv.merge(total_mv_per_year, on='年份', suffixes=('', '_total'))
group_weights['target_weight'] = group_weights['avg_total_mv'] / group_weights['avg_total_mv_total']
# 打印验证
print("\n3.3 各组市值占比（示例年份 2024）：")
# 筛选 2024 年的数据，并按占比从高到低排序
sample_weights = group_weights[group_weights['年份'] == 2024].sort_values('target_weight', ascending=False)
for _, row in sample_weights.iterrows():
    label = row['industry_label']
    pct = row['target_weight']
    ch_name = SUB_CATEGORY_NAMES.get(label, '')
    if ch_name:
        print(f"{label}({ch_name}): 占比={pct:.2%}")
    else:
        print(f"{label}({label}): 占比={pct:.2%}")

# 3.4 行业中性两轮迭代选股算法
neutral_selected_list = []
iteration_logs = []
# 对每一年的数据进行循环处理
for year in sorted(pool['年份'].unique()):
    year_pool = pool[pool['年份'] == year].copy()
    year_weights = group_weights[group_weights['年份'] == year]
    # 1. 第一轮初选
    top500 = year_pool.sort_values('patent_per_mv', ascending=False).head(500)
    # 2. 初始分配
    target_dict = dict(zip(year_weights['industry_label'], year_weights['target_weight']))
    allocation = {}
    for group in target_dict:
        allocation[group] = max(1, int(round(target_dict[group] * 100)))
    # 3. 迭代调整
    max_deviation = 999
    avg_deviation = 999
    for _ in range(20):
        selected = []
        for group, n in allocation.items():
            group_stocks = top500[top500['industry_label'] == group]
            selected.extend(group_stocks.head(n)[['股票代码', '年份', 'industry_label']].values.tolist())
        selected_df = pd.DataFrame(selected, columns=['股票代码', '年份', 'industry_label'])
        selected_df = selected_df.merge(year_pool[['股票代码', 'avg_total_mv']], on='股票代码')
        actual_weights = selected_df.groupby('industry_label')['avg_total_mv'].sum() / selected_df['avg_total_mv'].sum()
        deviations = {}
        for group in target_dict:
            actual = actual_weights.get(group, 0)
            target = target_dict[group]
            deviations[group] = actual - target
        max_deviation = max(abs(d) for d in deviations.values())
        avg_deviation = sum(abs(d) for d in deviations.values()) / len(deviations)
        if max_deviation < 0.03:
            break
        sorted_groups = sorted(deviations.items(), key=lambda x: abs(x[1]), reverse=True)
        for i in range(0, len(sorted_groups)-1, 2):
            g1, d1 = sorted_groups[i]
            g2, d2 = sorted_groups[i+1]
            if d1 > 0 and d2 < 0:
                allocation[g1] = max(1, allocation[g1] - 1)
                allocation[g2] = allocation[g2] + 1
    # 4. 补足机制
    selected_df = pd.DataFrame(selected, columns=['股票代码', '年份', 'industry_label'])
    if len(selected_df) < 100:
        remaining = year_pool[~year_pool['股票代码'].isin(selected_df['股票代码'])]
        remaining = remaining.sort_values('patent_per_mv', ascending=False)
        needed = 100 - len(selected_df)
        selected_df = pd.concat([selected_df, remaining.head(needed)[['股票代码', '年份', 'industry_label']]])
    selected_df['年份'] = year
    # 确保年份最多 100 只
    selected_df = selected_df.head(100)
    neutral_selected_list.append(selected_df)
    # 记录日志
    log_line = f"{year}: 初选{len(top500)}只+迭代后{len(selected_df)}只, 最大绝对偏差 = {max_deviation:.2%}, 平均绝对偏差 = {avg_deviation:.2%}"
    iteration_logs.append(log_line)
# 5. 最终合并
neutral_selected = pd.concat(neutral_selected_list).reset_index(drop=True)
print("\n3.4 行业中性两轮迭代选股算法")
for log in iteration_logs:
    print(log)
print(f"\n行业中性选股结果: {len(neutral_selected)} 条\n每年选股数:")
yearly_counts = neutral_selected['年份'].value_counts().sort_index()
print(yearly_counts)

# 3.5 行业中性组合回测
daily_neutral = daily.merge(neutral_selected[['股票代码', '年份']], on=['股票代码', '年份'], how='inner')
daily_neutral = daily_neutral[daily_neutral['年份'] == daily_neutral['交易日'].dt.year]
neutral_ret = daily_neutral.groupby('交易日').apply(
    lambda x: np.average(x['涨跌幅']/100, weights=x['总市值'])
).reset_index(name='neutral_ret')
neutral_ret = neutral_ret.set_index('交易日')

neutral_backtest = neutral_ret.join(benchmark[['指数日收益率']], how='inner')
neutral_backtest['指数日收益率'] = neutral_backtest['指数日收益率']/100
neutral_backtest = neutral_backtest.rename(columns={'指数日收益率': 'zz2000_ret'})
neutral_backtest['zz2000_nav'] = (1 + neutral_backtest['zz2000_ret']).cumprod()
neutral_backtest['neutral_nav'] = (1 + neutral_backtest['neutral_ret']).cumprod()
neutral_backtest['neutral_excess_ret'] = neutral_backtest['neutral_ret'] - neutral_backtest['zz2000_ret']

# 打印回测范围
print(f"\n行业中性组合日线数据: {len(daily_neutral)} 条")
print(f"行业中性回测数据：{len(neutral_backtest)} 个交易日")
print(f"日期范围：{neutral_backtest.index.min().date()} ~ {neutral_backtest.index.max().date()}")

print("\n回测数据前3行：")
print(neutral_backtest.head(3))

# 3.6 三策略绩效指标对比
print("\n3.6 三策略绩效指标对比")
# 1. 分别计算三个策略的指标字典
metrics_original = calc_metrics(backtest['port_ret'], backtest['zz2000_ret'])
metrics_neutral = calc_metrics(neutral_backtest['neutral_ret'], neutral_backtest['zz2000_ret'])
metrics_benchmark = calc_metrics(backtest['zz2000_ret'])
# 2. 提取指标列表，保证每个策略都有这些指标
keys = ['年化收益', '年化波动', '夏普比率', '最大回撤', '胜率', '信息比率', '超额年化']
# 3. 按行构建数据
data = []
for k in keys:
    row = {'指标': k}
    # 原始选股
    row['原始选股'] = metrics_original.get(k, '-')
    # 行业中性选股
    row['行业中性选股'] = metrics_neutral.get(k, '-')
    # 中证2000基准（只有前5个指标有值）
    row['中证2000'] = metrics_benchmark.get(k, '-')
    data.append(row)
df_metrics = pd.DataFrame(data)
# 打印表格
print(tabulate(df_metrics, headers='keys', tablefmt='psql', showindex=False))

# 3.7 Plotly 可视化（三策略对比）
print("\n3.7")
# 1. 准备行业分布对比数据（2024年，三个维度）
year_plot = 2024
# A. 中证2000基准的行业市值分布
# 直接从 pool 中取 2024 年候选池的原始行业分布
benchmark_dist = pool[pool['年份'] == year_plot].groupby('industry_label')['avg_total_mv'].sum()
benchmark_pct = benchmark_dist / benchmark_dist.sum()
# B. 原始选股的行业市值分布
# 从 backtest 中找到 2024 年原始选股的股票代码
selected_df = pd.DataFrame(selected, columns=['股票代码', '年份', 'industry_label'])
orig_codes_2024 = selected_df[selected_df['年份'] == year_plot]['股票代码'].tolist()
# 计算原始选股的市值占比
orig_dist = pool[(pool['年份'] == year_plot) & (pool['股票代码'].isin(orig_codes_2024))].groupby('industry_label')['avg_total_mv'].sum()
orig_pct = orig_dist / orig_dist.sum()
# C. 行业中性选股的行业市值分布
# 从 neutral_selected 中找到 2024 年中性选股的股票代码
neut_codes_2024 = neutral_selected[neutral_selected['年份'] == year_plot]['股票代码'].tolist()
neut_dist = pool[(pool['年份'] == year_plot) & (pool['股票代码'].isin(neut_codes_2024))].groupby('industry_label')['avg_total_mv'].sum()
neut_pct = neut_dist / neut_dist.sum()
# 合并所有行业标签（保证柱状图X轴一致）
all_labels = sorted(set(benchmark_pct.index) | set(orig_pct.index) | set(neut_pct.index))
# 2. 创建画布（3行1列）
fig = make_subplots(rows=3, cols=1, shared_xaxes=False,  # 第3个图是柱状图，不用共享X轴
    vertical_spacing=0.12, row_heights=[0.45, 0.25, 0.30],
    subplot_titles=("净值对比", "超额收益对比", "行业分布对比 (2024年)"))

# 第1个图：净值对比
fig.add_trace(go.Scatter(x=backtest.index, y=backtest['port_nav'], mode='lines', 
    name='原始选股',line=dict(color="#fa2525", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=neutral_backtest.index, y=neutral_backtest['neutral_nav'],
    mode='lines', name='行业中性', line=dict(color="#ffc30e", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=backtest.index, y=backtest['zz2000_nav'], mode='lines', 
    name='中证2000', line=dict(color="#1d30af", width=2)), row=1, col=1)

# 第2个图：超额收益对比
# 原始超额净值
fig.add_trace(go.Scatter(x=backtest.index, y=backtest['excess_nav'], mode='lines', 
    name='原始超额', line=dict(color="#fa2525", width=2), showlegend=False), 
    row=2, col=1)
# 行业中性超额净值
fig.add_trace(go.Scatter(x=neutral_backtest.index, 
    y=neutral_backtest['neutral_nav'] / neutral_backtest['zz2000_nav'], mode='lines', 
    name='行业中性超额', line=dict(color="#ffbc20", width=2), showlegend=False), 
    row=2, col=1)
# 在 y=1.0 处画虚线
fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)

# 第3个图：行业分布对比（分组柱状图）
# 1. 先找到所有行业标签（去掉 X 类）
all_labels = sorted(set(benchmark_pct.index) | set(neut_pct.index) | set(orig_pct.index))
all_labels = [l for l in all_labels if l != 'X']
# 2. 把 X 轴变成数字，手动错开位置
x_positions = np.arange(len(all_labels))
width = 0.25  # 每一根柱子的宽度
# 3. 准备三组数据
bench_vals = [benchmark_pct.get(l, 0) for l in all_labels]
neut_vals = [neut_pct.get(l, 0) for l in all_labels]
orig_vals = [orig_pct.get(l, 0) for l in all_labels]
# 4. 分别添加三组柱状图，每个向左/右偏移宽度
fig.add_trace(go.Bar(x=x_positions - width, y=bench_vals, name='中证2000',
    marker=dict(color='#d62728'), width=width), row=3, col=1)
fig.add_trace(go.Bar(x=x_positions + width, y=neut_vals, name='行业中性',
    marker=dict(color='#ff7f0e'), width=width), row=3, col=1)
fig.add_trace(go.Bar(x=x_positions, y=orig_vals, name='原始选股',
    marker=dict(color='#1f77b4'), width=width), row=3, col=1)
# 5. 最后把 X 轴的数字标签换回原本的行业名称
fig.update_xaxes(tickvals=x_positions, ticktext=all_labels, row=3, col=1)
# 6. 设置Y轴为百分比
fig.update_yaxes(title_text="占比", row=3, col=1, tickformat='.0%')

# 图表美化设置
# 更新Y轴格式
fig.update_yaxes(title_text="净值", row=1, col=1)
fig.update_yaxes(title_text="超额净值", row=2, col=1)
fig.update_yaxes(title_text="占比", row=3, col=1, tickformat='.0%')
# 第3个图设置分组柱状图模式
fig.update_layout(barmode='group', bargap=0.15)
# 设置图例交互（防止只显示一个）
fig.update_layout(height=900, width=1100, legend=dict(orientation="h", yanchor="bottom", 
    y=1.02, xanchor="right", x=1, itemclick='toggle', itemdoubleclick='toggle'),
    margin=dict(l=50, r=50, t=80, b=50))
fig.show()

# 3.8 数据导出
print('\n3.8')
backtest['strategy'] = '原始选股'
neutral_backtest['strategy'] = '行业中性'
neutral_backtest['zz2000_nav'] = (1 + neutral_backtest['zz2000_ret']).cumprod()

all_strategies = pd.concat([
    backtest[['port_ret', 'port_nav', 'zz2000_ret', 'zz2000_nav', 'strategy']].reset_index(),
    neutral_backtest[['neutral_ret', 'neutral_nav', 'zz2000_ret', 'zz2000_nav', 'strategy']].reset_index().rename(
        columns={'neutral_ret': 'port_ret', 'neutral_nav': 'port_nav'}
    )
])
print(all_strategies.head())
all_strategies.to_parquet('D:/Limmo/Python数分/实验四/return.parquet')
print("数据已导出至 D:/Limmo/Python数分/实验四/return.parquet")

daily: 18685210 行, 6 列
patent: 6096734 行, 5 列
finance: 76262 行, 7 列
专利去重: 6096734 → 5032442（去除1064292条重复，17.5%）
专利缺失值过滤: 5032442 → 5032441（去除1条）
日线缺失值过滤: 18685210 → 17372540（去除1312670条）

中证2000成分股：2000只
中证2000代码集：2000个
ST股：294只
ST代码集：294个
中证2000日线：3015条

财报因子：56933条
公司成立日期: 2606条

1.4 数据质量检查报告

1. 数据规模
daily: 17372540 行, 6 列
patent: 5032441 行, 5 列

2. 关键字段时间范围
daily.交易日: 1990-12-19 至 2026-02-27
patent.申请年份: 1985.0 至 2025.0

3. 关键字段缺失率
daily.收盘价:缺失率 0.00%
daily.总市值:缺失率 0.00%
daily.涨跌幅:缺失率 0.00%
patent.关联股票代码：缺失率 0.00%

4. 重复记录检查
daily: 总行数=17372540, 唯一行=17372540, 重复行: 0 (0.00%)
patent: 总行数=5032441, 唯一行=5032441, 重复行: 0 (0.00%)

5. 关键数值字段异常值
daily ['收盘价'<=0]: 0/17372540(0.00%)
daily ['涨跌幅'>20%]: 13421/17372540(0.08%)

6. 候选池交叉覆盖 (中证2000与日线、专利交集)
中证2000成分股总数量: 2000
同时出现在日线数据中的数量: 2000 (100.00%)
同时出现在专利数据中的数量: 1859 (92.95%)
同时出现在日线数据和专利数据中的数量: 1859 (92.95%)


1.5 专利因子与面板数据构建
专利聚合：51181 条 (股票×年份)
股票数：5417
波动率+市值+收益率: 51178 条

1.6 合并面板与候选池标记
合并面板: 51181 条
中证2000+非ST: 17462 条
股票数: 5417
年份: 201


3.1 行业分类映射 (CAPCO 二级分类)
原始表行数: 56933, 有行业代码的行数: 45973
行业映射: 5436 只股
次类代码分布:
sub_category
CL    1034
CI     823
I      451
CE     367
CF     332
CK     273
F      211
CA     197
CG     159
D      141
K      134
J      133
G      127
CH     123
E      115
M      112
CB     112
N      101
B       84
CD      82
L       72
R       65
A       52
CC      42
CM      38
Q       17
S       14
H       12
P       12
O        1
Name: count, dtype: int64 None

3.2 合并后分组标签 (示例年份2024):
CL(电气计算机仪器仪表): 372只
C: 309只
CI(金属): 295只
I(信息传输、软件和信息技术服务业): 134只
CE(石油化学): 129只
CK(汽车船舶航空航天): 97只
M(科学研究和技术服务业): 43只
F(批发和零售业): 40只
D(电力、热力、燃气及水生产和供应业): 36只
E(建筑业): 25只
K(房地产业): 20只
L(租赁和商务服务业): 19只
G(交通运输、仓储和邮政业): 18只
A(农、林、牧、渔业): 11只
R(文化、体育和娱乐业): 11只
B(采矿业): 8只
Q(卫生和社会工作): 4只
S(综合): 4只
H(住宿和餐饮业): 2只
P(教育): 2只

3.3 各组市值占比（示例年份 2024）：
CL(电气计算机仪器仪表): 占比=21.76%
C(C): 占比=17.99%
CI(金属): 占比=16.69%
CE(石油化学): 占比=7.53%
I(信息传输、软件和信息技术服务业): 占比=7.52%
X(X): 占比=7.17%
CK(汽车船舶航空航天): 占比=5.93%
F(批发和零售业): 占比=2.56%
M(科学研究和技术服务业): 占比=2.


3.8
         交易日  port_ret  port_nav  zz2000_ret  zz2000_nav strategy
0 2014-01-02  0.013951  1.013951    0.000000    1.000000     原始选股
1 2014-01-03 -0.001511  1.012420   -0.003414    0.996586     原始选股
2 2014-01-06 -0.029918  0.982130   -0.029734    0.966954     原始选股
3 2014-01-07  0.013577  0.995465    0.008028    0.974716     原始选股
4 2014-01-08  0.009935  1.005355    0.005804    0.980373     原始选股
数据已导出至 D:/Limmo/Python数分/实验四/return.parquet
